# Lab 44 (solution): Hardening the signals for real traffic

Reference implementation. The three [Lab 42](../42-hardening-operations/) signals, hardened for the conditions live traffic actually creates: the notifier gains retries, rate limiting, and dedup/cooldown; the drift baseline moves off the circular prototype set onto a held-out clean reference; and the canary suite grows to cover named failure modes with a corpus-change review.

The hardened scripts (`notify.py`, `record_baseline.py`, `canary.py`) and the new `reference_sample.jsonl` live in the [operating-the-loop](../41-operating-the-loop/) toolkit; the workflows are updated in place.

## Step 0: Setup

In [ ]:
import json
import pathlib
import sys
# Lab 44 hardens the three Lab 42 signals for real traffic. The scripts live in the
# operating-the-loop toolkit; point at it.
loop = pathlib.Path.cwd().parent / "41-operating-the-loop"
sys.path.insert(0, str(loop))
print("hardening signals in:", loop.name)

## Step 1: Notifier retries, rate limit, cooldown (item 1)

In [ ]:
from notify import deliver, format_alert, to_slack, send_with_retry, rate_limited, in_cooldown, record_send
from urllib.error import URLError
# Item 1: before live traffic the notifier needs to survive a flaky webhook (retry),
# a regression storm (rate limit), and its own repetition (dedup/cooldown).

# retry: fail twice, then succeed - no lost page
calls={"n":0}
def flaky():
    calls["n"]+=1
    if calls["n"] < 3:
        raise URLError("transient")
    return "posted (200)"
print("retry result:", send_with_retry(flaky, retries=3, sleep=lambda s: None), "after", calls["n"], "tries")

# cooldown: the same incident does not re-page while you are already on it
p=format_alert("judged_faithfulness", 0.55, 0.764)
state={}
s1,state=deliver(to_slack(p), p, url=None, state=state, now=0.0, sleep=lambda s: None)
s2,state=deliver(to_slack(p), p, url=None, state=state, now=600.0, sleep=lambda s: None)
print("first:", s1)
print("re-fire 10 min later:", s2)

# rate limit: a burst is capped
burst={"window":[0,1,2,3,4]}
print("6th alert in the window blocked?", rate_limited(5.0, burst, max_per_window=5, window_s=3600))

## Step 2: A held-out drift baseline (item 2)

Stop measuring against your own training data.

In [ ]:
from record_baseline import compute_baseline
# Item 2: the Lab 42 baseline measured confidence on the PROTOTYPE trainset - circular,
# because a model is over-confident on its own training data. A held-out clean reference
# (reference_sample.jsonl: realistic phrasings the model never trained on) gives an
# REALISTIC band. The numbers below stand in for what you'd get from the embedder.
prototype = compute_baseline([0.93, 0.95, 0.91, 0.94, 0.92])   # confidence ON the trainset
heldout   = compute_baseline([0.80, 0.78, 0.83, 0.76, 0.79])   # confidence on held-out phrasing
print("prototype-set band (optimistic):", prototype)
print("held-out band (realistic):        ", heldout)
print(f"\nThe held-out mean ({heldout['mean']}) sits below the prototype mean ({prototype['mean']}).")
print("Drift judged against the optimistic band under-fires; the realistic band catches real sag.")
with open(loop / "reference_sample.jsonl") as f:
    ref = [json.loads(line) for line in f]
print(f"reference_sample.jsonl: {len(ref)} held-out queries, none verbatim in the trainset.")

## Step 3: Failure-mode canaries + corpus review (item 3)

In [ ]:
from canary import load_canaries, review_status, canaries_needing_review
from collections import Counter
# Item 3: grow the canary set to cover named failure modes, and review canaries when the
# corpus changes (their reference answers may go stale).
cans=load_canaries()
print("failure-mode coverage:", dict(Counter(c["failure_mode"] for c in cans)))

# corpus-change review: a changed fingerprint flags exactly the corpus-dependent canaries
rs=review_status(cans, recorded_fp="OLD", current_fp="NEW")
print(f"\ncorpus changed -> revalidate {len(rs['to_review'])} canaries (the corpus-dependent ones)")
free=[c["query"] for c in cans if not c.get("corpus_refs")]
print(f"corpus-free canaries (parametric / pure refusal) unaffected: {len(free)}")
print("unchanged corpus ->", review_status(cans, recorded_fp="SAME", current_fp="SAME")["to_review"], "(nothing to review)")

## Step 4: The hardened cadence

In [ ]:
# The hardened cadence (workflows updated in place):
#   rag-faithfulness-nightly  notify persists cooldown state across runs (actions/cache +
#                             --state-file), so a standing incident pages once, not nightly
#   rag-drift-check           + a non-blocking canary corpus-review step
#   rag-maintenance-loop      promote records the baseline on the HELD-OUT reference
print("Cooldown survives across nightly runs; the corpus-review step names stale canaries;")
print("the promote step records a realistic, held-out baseline.")

## Step 5: What this buys

In [ ]:
# What 'before high volume' hardening buys:
#  - retries: a transient webhook failure does not silently drop a page;
#  - rate limit + cooldown: a regression does not become an alert storm or nightly nag;
#  - held-out baseline: the drift band reflects real phrasing, not training-set optimism;
#  - failure-mode canaries + corpus review: you test the breaks you actually fear, and you
#    know which canaries to revalidate the moment the corpus changes.
print("A signal you can point at production is one that fails loudly, alerts sparingly,")
print("measures against reality, and tells you when its own assumptions went stale.")

## What you built

The production-traffic hardening of the three signals: `notify.py` now retries transient delivery failures with backoff, rate-limits a burst, and suppresses a re-fire of the same incident within a cooldown (persisted across runs via a state file); `record_baseline.py` measures the drift band on a held-out clean reference instead of the trainset the model is over-confident on; and `canary.py` covers named failure modes and flags the corpus-dependent canaries for revalidation whenever the corpus fingerprint changes.

**Where this simplifies:** the rate limiter is a single-process fixed window (a real fleet needs a shared store); the cooldown state is a local JSON file (cache it or use a real store in CI); the held-out reference is sixteen curated queries (collect a larger clean sample from real traffic); the corpus fingerprint is a content hash of the whole corpus (a finer-grained, per-doc map would tell you *which* canaries more precisely).

Next: [Lab 45](../45-anchoring-the-consensus/) makes the same move on the evaluation side — stop measuring the judge against an unanchored majority vote.